
# Executive Communication Optimizer & Tone Analyzer

## Objetivos de Aprendizaje

In this notebook you'll learn how to:
1. Build an **executive messaging warehouse** with Snowflake SQL
2. Score communications with `AI_SENTIMENT()`
3. Generar executive-ready themes via `AI_COMPLETE()`
4. Summarize risk statements using `AI_SUMMARIZE()`
5. Deliver dynamic dashboards for global communications teams

---

## What You'll Build
- AI-enriched sentiment baseline for speeches, press releases, and investor calls
- LLM theme extraction + recommended actions for negative communications
- Multi-dimensional SQL analytics (clarity, cadence, consistency, stakeholder reach)
- Streamlit dashboard with dependency checks and CSV export

**Estimated runtime:** ~12 minutes on `SMALL` warehouse


---

In [ ]:

-- Create environment
CREATE DATABASE IF NOT EXISTS EXEC_COMM_DB;
CREATE SCHEMA IF NOT EXISTS EXEC_COMM_DB.PUBLIC;
USE SCHEMA EXEC_COMM_DB.PUBLIC;

CREATE WAREHOUSE IF NOT EXISTS COMPUTE_WH 
    WITH WAREHOUSE_SIZE = 'SMALL'
    AUTO_SUSPEND = 60
    AUTO_RESUME = TRUE;

USE WAREHOUSE COMPUTE_WH;

SELECT 
    CURRENT_DATABASE() as database_name,
    CURRENT_SCHEMA() as schema_name,
    CURRENT_WAREHOUSE() as warehouse_name,
    'Executive Communication Optimizer Environment Ready!' as status;


In [ ]:
-- Drop existing tables for idempotent rerun
DROP TABLE IF EXISTS communication_themes;
DROP TABLE IF EXISTS communication_sentiment;
DROP TABLE IF EXISTS communication_metrics;
DROP TABLE IF EXISTS executive_communications;


In [ ]:
-- Create sentiment analysis table
CREATE TABLE IF NOT EXISTS communication_sentiment (
    sentiment_id STRING,
    message_id STRING,
    exec_name STRING,
    message_type STRING,
    audience STRING,
    topic STRING,
    sentiment_score FLOAT,
    sentiment_label STRING,
    sentiment_magnitude FLOAT,
    analyzed_timestamp TIMESTAMP_NTZ
);

CREATE TABLE IF NOT EXISTS communication_themes (
    theme_id STRING,
    message_id STRING,
    exec_name STRING,
    topic STRING,
    key_theme STRING,
    recommended_action STRING,
    executive_summary STRING,
    severity_flag STRING,
    generated_timestamp TIMESTAMP_NTZ
);

SELECT 'analysis tables ready' AS status;


In [ ]:
-- Create base communications table
CREATE OR REPLACE TABLE executive_communications (
    message_id STRING,
    exec_name STRING,
    exec_role STRING,
    message_type STRING,
    audience STRING,
    topic STRING,
    channel STRING,
    region STRING,
    language STRING,
    content STRING,
    created_timestamp TIMESTAMP_NTZ,
    metadata VARIANT,
    ingest_timestamp TIMESTAMP_NTZ,
    PRIMARY KEY (message_id)
);

SELECT 'executive_communications ready' AS status;


---

## Paso 2: Define Data Structure

### Tables

1. **`exec_communications`** - Primary data source
2. **`communication_analysis`** - Analysis results
3. **`tone_metrics`** - Aggregated insights

### Data Flow

1. **Ingest** → Collect data from sources
2. **Process** → Apply AI functions
3. **Analyze** → Calculate metrics
4. **Visualize** → Present insights

In [ ]:
-- Calculate daily metrics
CREATE OR REPLACE TABLE communication_metrics AS
SELECT 
    'METRIC_' || LPAD(ROW_NUMBER() OVER (ORDER BY r.created_timestamp::DATE)::VARCHAR, 8, '0') AS metric_id,
    r.created_timestamp::DATE AS metric_date,
    'daily_avg_sentiment' AS metric_name,
    ROUND(AVG(s.sentiment_score), 3) AS metric_value,
    CASE 
        WHEN AVG(s.sentiment_score) > LAG(AVG(s.sentiment_score)) OVER (ORDER BY r.created_timestamp::DATE) THEN 'up'
        WHEN AVG(s.sentiment_score) < LAG(AVG(s.sentiment_score)) OVER (ORDER BY r.created_timestamp::DATE) THEN 'down'
        ELSE 'stable'
    END AS trend,
    CURRENT_TIMESTAMP() AS generated_timestamp
FROM executive_communications r
JOIN communication_sentiment s ON r.message_id = s.message_id
GROUP BY r.created_timestamp::DATE;

SELECT * FROM communication_metrics ORDER BY metric_date DESC LIMIT 10;


---

## Paso 3: Generar Datos Sintéticos Data

### Qué Vamos a Crear

- **1,000 records** for demonstration
- **Realistic content** matching use case
- **Last 30 days** of data
- **Multiple sources** and categories

### Synthetic Data Approach

Using Snowflake's `GENERATOR()` and `UNIFORM()` functions to create realistic test data.

In [ ]:

-- Generar datos sintéticos executive communications
INSERT INTO executive_communications
WITH execs AS (
    SELECT * FROM (VALUES
        ('Emma Sørensen','Chief Communications Officer'),
        ('Lars Jensen','Investor Relations Lead'),
        ('Maria Chen','SVP Public Affairs'),
        ('Noah Patel','Chief Scientific Officer'),
        ('Ava Lindholm','VP Corporate Strategy')
    ) AS t(exec_name, exec_role)
),
message_types AS (SELECT * FROM (VALUES ('Press Release'), ('Investor Call'), ('Town Hall'), ('Keynote'), ('Board Brief'))),
audiences AS (SELECT * FROM (VALUES ('Investors'), ('Employees'), ('Regulators'), ('Media'), ('Healthcare Providers'))),
topics AS (SELECT * FROM (VALUES ('Metabolic Health'), ('Pipeline Update'), ('Sustainability'), ('Access & Pricing'), ('Manufacturing Capacity'))),
channels AS (SELECT * FROM (VALUES ('Earnings Call'), ('LinkedIn Live'), ('Press Hub'), ('Internal Network'), ('Media Briefing'))),
languages AS (SELECT * FROM (VALUES ('EN'), ('DA'), ('DE'), ('FR'))),
regions AS (SELECT * FROM (VALUES ('North America'), ('Europe'), ('Asia Pacific'))),
content_templates AS (
    SELECT * FROM (VALUES
        ('Thank you for joining. Today we reinforced our commitment to innovation while addressing stakeholder concerns about access and affordability.'),
        ('Our executive remarks highlighted milestone results, upcoming regulatory submissions, and community partnerships that expand treatment availability.'),
        ('We recognize recent feedback about supply reliability and are implementing additional transparency and executive availability to rebuild confidence.'),
        ('During this session we celebrated metabolic health outcomes, but we also acknowledged the cost pressures and pledged clearer communication to payers.'),
        ('We emphasized scientific leadership, collaborative tone, and alignment with Bayer values to guide employees through strategic change.')
    ) AS t(template)
)
SELECT 
    'MSG_' || LPAD(ROW_NUMBER() OVER (ORDER BY g.seq)::VARCHAR, 8, '0') AS message_id,
    e.exec_name,
    e.exec_role,
    mt.column1 AS message_type,
    au.column1 AS audience,
    tp.column1 AS topic,
    ch.column1 AS channel,
    rg.column1 AS region,
    lang.column1 AS language,
    CONCAT(ct.template, ' (', tp.column1, ' update delivered via ', ch.column1, ')') AS content,
    DATEADD('day', -FLOOR(UNIFORM(0, 45, RANDOM())), CURRENT_TIMESTAMP()) AS created_timestamp,
    OBJECT_CONSTRUCT(
        'message_type', mt.column1,
        'audience', au.column1,
        'topic', tp.column1,
        'channel', ch.column1,
        'region', rg.column1,
        'language', lang.column1,
        'cta', ARRAY_CONSTRUCT('Reassure stakeholders','Highlight data','Explain pricing','Clarify supply','Celebrate progress')[FLOOR(UNIFORM(0,5,RANDOM()))]
    ) AS metadata,
    CURRENT_TIMESTAMP() AS ingest_timestamp
FROM (
    SELECT ROW_NUMBER() OVER (ORDER BY SEQ4()) AS seq
    FROM TABLE(GENERATOR(ROWCOUNT => 700))
) g
JOIN execs e ON TRUE
JOIN message_types mt ON TRUE
JOIN audiences au ON TRUE
JOIN topics tp ON TRUE
JOIN channels ch ON TRUE
JOIN regions rg ON TRUE
JOIN languages lang ON TRUE
JOIN content_templates ct ON TRUE
WHERE UNIFORM(0,1,RANDOM()) < 0.35
QUALIFY ROW_NUMBER() OVER (ORDER BY created_timestamp DESC) <= 320;

SELECT 
    COUNT(*) AS total_messages,
    COUNT(DISTINCT exec_name) AS executives,
    MIN(created_timestamp) AS earliest_message,
    MAX(created_timestamp) AS latest_message
FROM executive_communications;


---

## Paso 3: Load Synthetic Executive Communications

We generate a balanced portfolio of investor calls, town halls, keynotes, and press releases with realistic metadata (audience, topic, channel, region). This mirrors the daily communications tracking feed.

In [ ]:

-- Apply Cortex AI analysis to communications
-- Verified via Snowflake docs 2025-01: AI_SENTIMENT returns OBJECT
CREATE OR REPLACE TABLE communication_sentiment AS
WITH ranked_messages AS (
    SELECT 
        message_id,
        exec_name,
        message_type,
        audience,
        topic,
        content,
        ROW_NUMBER() OVER (ORDER BY created_timestamp DESC, message_id) AS row_id
    FROM executive_communications
)
SELECT 
    'SENT_' || LPAD(row_id::VARCHAR, 8, '0') AS sentiment_id,
    message_id,
    exec_name,
    message_type,
    audience,
    topic,
    AI_SENTIMENT(content)['categories'][0]['sentiment']::STRING as sentiment_label_raw,
    -- Convert sentiment to numeric score (Use Case 28 pattern)
    CASE AI_SENTIMENT(content)['categories'][0]['sentiment']::STRING
        WHEN 'positive' THEN 1.0
        WHEN 'neutral' THEN 0.0
        WHEN 'negative' THEN -1.0
        WHEN 'mixed' THEN -0.5
        ELSE -1.0
    END AS sentiment_score,
    CASE sentiment_label_raw
        WHEN 'positive' THEN 'Highly Positive'
        WHEN 'neutral' THEN 'Neutral'
        WHEN 'negative' THEN 'Negative'
        WHEN 'mixed' THEN 'Negative'
        ELSE 'Highly Negative'
    END AS sentiment_label,
    ABS(CASE AI_SENTIMENT(content)['categories'][0]['sentiment']::STRING
        WHEN 'positive' THEN 1.0
        WHEN 'neutral' THEN 0.0
        WHEN 'negative' THEN -1.0
        WHEN 'mixed' THEN -0.5
        ELSE -1.0
    END) AS sentiment_magnitude,
    CURRENT_TIMESTAMP() AS analyzed_at
FROM ranked_messages
QUALIFY row_id <= 800;

SELECT 
    sentiment_label,
    COUNT(*) AS message_count,
    ROUND(AVG(sentiment_score), 3) AS avg_sentiment,
    ROUND(AVG(sentiment_magnitude), 3) AS avg_magnitude
FROM communication_sentiment
GROUP BY sentiment_label
ORDER BY avg_sentiment DESC;

---

## Paso 5: Extract Themes with `AI_COMPLETE()` + `SUMMARIZE()`

**What:** Enrich low-sentiment communications with LLM-generated themes, recommended actions, and executive summaries.

**Why it matters:** Converts raw scores into specific coaching moments for communications teams.

**Technical details:** `AI_COMPLETE()` produces a single theme + action, while `AI_SUMMARIZE()` generates a 1-sentence brief saved in `communication_themes`.

In [ ]:

-- LLM theme extraction for negative/neutral communications
CREATE OR REPLACE TABLE communication_themes AS
WITH prioritized AS (
    SELECT 
        s.sentiment_id,
        s.message_id,
        s.exec_name,
        s.topic,
        s.sentiment_score,
        s.sentiment_label,
        ec.content,
        ROW_NUMBER() OVER (ORDER BY s.sentiment_score) AS row_id
    FROM communication_sentiment s
    JOIN executive_communications ec ON s.message_id = ec.message_id
    WHERE s.sentiment_score < 0.15
)
SELECT 
    'THEME_' || LPAD(row_id::VARCHAR, 8, '0') AS theme_id,
    message_id,
    exec_name,
    topic,
    AI_COMPLETE(
        'mistral-large2',
        'You are a communications coach. Classify the executive message into ONE theme (Investor Clarity, Supply Risk, Pricing Pressure, Scientific Leadership, Employee Alignment, Regulatory Focus). Return only the theme name. Message: ' || content
    ) AS key_theme,
    AI_COMPLETE(
        'mistral-large2',
        'Recommend a concise next action (max 20 words) to improve this message for stakeholders. Message: ' || content
    ) AS recommended_action,
    SNOWFLAKE.CORTEX.SUMMARIZE(content) AS executive_summary,
    CASE 
        WHEN sentiment_score <= -0.4 THEN 'High Risk'
        WHEN sentiment_score <= -0.15 THEN 'Moderate Risk'
        ELSE 'Monitor'
    END AS severity_flag,
    CURRENT_TIMESTAMP() AS generated_timestamp
FROM prioritized
QUALIFY row_id <= 150;

SELECT key_theme, severity_flag, COUNT(*) AS issue_count
FROM communication_themes
GROUP BY key_theme, severity_flag
ORDER BY issue_count DESC;


In [ ]:

-- Message Clarity & Impact Analysis
WITH message_metrics AS (
    SELECT 
        r.exec_name,
        r.message_type,
        r.audience,
        COUNT(*) AS message_count,
        ROUND(AVG(s.sentiment_score), 3) AS avg_sentiment,
        ROUND(STDDEV(s.sentiment_score), 3) AS sentiment_consistency,
        ROUND(AVG(s.sentiment_magnitude), 3) AS avg_intensity,
        ROUND(AVG(LENGTH(r.content)), 0) AS avg_message_length,
        SUM(CASE WHEN s.sentiment_score >= 0.3 THEN 1 ELSE 0 END) AS positive_messages,
        SUM(CASE WHEN s.sentiment_score BETWEEN -0.3 AND 0.3 THEN 1 ELSE 0 END) AS neutral_messages,
        SUM(CASE WHEN s.sentiment_score <= -0.3 THEN 1 ELSE 0 END) AS negative_messages
    FROM executive_communications r
    JOIN communication_sentiment s ON r.message_id = s.message_id
    GROUP BY r.exec_name, r.message_type, r.audience
)
SELECT 
    exec_name,
    message_type,
    audience,
    message_count,
    avg_sentiment,
    sentiment_consistency,
    avg_intensity,
    avg_message_length,
    positive_messages,
    neutral_messages,
    negative_messages,
    ROUND(positive_messages * 100.0 / NULLIF(message_count,0), 1) AS pct_positive,
    CASE 
        WHEN avg_sentiment >= 0.55 AND sentiment_consistency <= 0.25 THEN '🌟 High Impact'
        WHEN avg_sentiment >= 0.25 THEN '✅ Effective'
        WHEN avg_sentiment >= 0 THEN '📊 Moderate'
        ELSE '⚠️ Needs Review'
    END AS impact_rating,
    CASE 
        WHEN avg_message_length <= 220 AND sentiment_consistency <= 0.25 THEN '💎 Clear & Concise'
        WHEN avg_message_length <= 320 THEN '👍 Good Clarity'
        WHEN avg_message_length <= 520 THEN '📝 Moderate Length'
        ELSE '📄 Detailed (consider brevity)'
    END AS clarity_assessment
FROM message_metrics
ORDER BY avg_sentiment DESC, message_count DESC;


---

## Paso 7: Stakeholder Communication Patterns

### Qué Vamos a Hacer

Analyze **how different executives communicate with specific stakeholder groups** (investors, employees, media, etc.).

### Why This Matters

- **Tailored messaging**: Understand what works for each audience
- **Consistency check**: Ensure aligned messaging across stakeholders
- **Gap identification**: Find under-communicated stakeholder groups

### Analysis Focus

- Communication frequency by stakeholder type
- Sentiment patterns per audience
- Executive coverage across stakeholder groups

In [ ]:

-- Stakeholder Communication Patterns
WITH stakeholder_analysis AS (
    SELECT 
        r.audience AS stakeholder_group,
        r.exec_name,
        r.message_type,
        COUNT(*) AS communication_count,
        ROUND(AVG(s.sentiment_score), 3) AS avg_sentiment,
        ROUND(AVG(s.sentiment_magnitude), 3) AS avg_intensity,
        MIN(r.created_timestamp::DATE) AS first_communication,
        MAX(r.created_timestamp::DATE) AS latest_communication,
        DATEDIFF('day', MIN(r.created_timestamp), MAX(r.created_timestamp)) AS days_span
    FROM executive_communications r
    JOIN communication_sentiment s ON r.message_id = s.message_id
    WHERE r.audience IS NOT NULL
    GROUP BY r.audience, r.exec_name, r.message_type
),
stakeholder_summary AS (
    SELECT 
        stakeholder_group,
        COUNT(DISTINCT exec_name) AS executives_communicating,
        SUM(communication_count) AS total_communications,
        ROUND(AVG(avg_sentiment), 3) AS group_avg_sentiment,
        ROUND(AVG(avg_intensity), 3) AS group_avg_intensity,
        MAX(latest_communication) AS most_recent
    FROM stakeholder_analysis
    GROUP BY stakeholder_group
)
SELECT 
    sa.stakeholder_group,
    sa.exec_name,
    sa.message_type,
    sa.communication_count,
    sa.avg_sentiment,
    sa.avg_intensity,
    sa.latest_communication,
    sa.days_span AS engagement_span_days,
    ss.total_communications AS group_total,
    ss.executives_communicating,
    ROUND(sa.communication_count * 100.0 / NULLIF(ss.total_communications,0), 1) AS pct_of_group_total,
    CASE 
        WHEN sa.communication_count >= 10 AND sa.days_span >= 25 THEN '🔥 Highly Engaged'
        WHEN sa.communication_count >= 5 THEN '✅ Active'
        WHEN sa.communication_count >= 2 THEN '📊 Moderate'
        ELSE '💤 Limited'
    END AS engagement_level,
    CASE 
        WHEN sa.avg_sentiment > ss.group_avg_sentiment + 0.2 THEN '📈 Above Group Avg'
        WHEN sa.avg_sentiment < ss.group_avg_sentiment - 0.2 THEN '📉 Below Group Avg'
        ELSE '➡️ Aligned with Group'
    END AS sentiment_comparison
FROM stakeholder_analysis sa
JOIN stakeholder_summary ss USING (stakeholder_group)
ORDER BY sa.stakeholder_group, sa.communication_count DESC;


---

## Paso 8: Communication Effectiveness Metrics

### Qué Vamos a Hacer

Measure **communication effectiveness** through engagement, response rates, and action taken.

### Why This Matters

- **ROI of communications**: Know what's working
- **Optimize strategy**: Focus on high-performing patterns
- **Executive coaching**: Provide data-driven feedback

### Effectiveness Indicators

- Sentiment trends over time
- Message-to-action conversion
- Stakeholder engagement patterns
- Communication frequency optimization

In [ ]:

-- Communication Effectiveness Metrics
WITH daily_effectiveness AS (
    SELECT 
        r.created_timestamp::DATE AS created_date,
        r.exec_name,
        COUNT(*) AS messages_sent,
        ROUND(AVG(s.sentiment_score), 3) AS daily_avg_sentiment,
        ROUND(AVG(s.sentiment_magnitude), 3) AS daily_avg_intensity,
        LAG(ROUND(AVG(s.sentiment_score), 3)) OVER (
            PARTITION BY r.exec_name ORDER BY r.created_timestamp::DATE
        ) AS prev_day_sentiment
    FROM executive_communications r
    JOIN communication_sentiment s ON r.message_id = s.message_id
    GROUP BY r.created_timestamp::DATE, r.exec_name
),
exec_summary AS (
    SELECT 
        exec_name,
        COUNT(*) AS total_days_active,
        SUM(messages_sent) AS total_messages,
        ROUND(AVG(messages_sent), 1) AS avg_messages_per_day,
        ROUND(AVG(daily_avg_sentiment), 3) AS overall_avg_sentiment,
        ROUND(STDDEV(daily_avg_sentiment), 3) AS sentiment_volatility,
        SUM(CASE WHEN daily_avg_sentiment > prev_day_sentiment + 0.1 THEN 1 ELSE 0 END) AS improving_days,
        SUM(CASE WHEN daily_avg_sentiment < prev_day_sentiment - 0.1 THEN 1 ELSE 0 END) AS declining_days,
        SUM(CASE WHEN ABS(daily_avg_sentiment - prev_day_sentiment) <= 0.1 THEN 1 ELSE 0 END) AS stable_days
    FROM daily_effectiveness
    GROUP BY exec_name
)
SELECT 
    exec_name,
    total_days_active,
    total_messages,
    avg_messages_per_day,
    overall_avg_sentiment,
    sentiment_volatility,
    improving_days,
    declining_days,
    stable_days,
    ROUND(improving_days * 100.0 / NULLIF(improving_days + declining_days + stable_days, 0), 1) AS pct_improving,
    ROUND((overall_avg_sentiment + 1) * 50, 1) AS effectiveness_score,
    CASE 
        WHEN sentiment_volatility <= 0.2 THEN '🎯 Highly Consistent'
        WHEN sentiment_volatility <= 0.4 THEN '✅ Consistent'
        WHEN sentiment_volatility <= 0.6 THEN '📊 Variable'
        ELSE '⚠️ Inconsistent'
    END AS consistency_rating,
    CASE 
        WHEN avg_messages_per_day >= 3 THEN '🔥 Very Active'
        WHEN avg_messages_per_day >= 1.5 THEN '✅ Active'
        WHEN avg_messages_per_day >= 0.5 THEN '📊 Moderate'
        ELSE '💤 Low Activity'
    END AS activity_level,
    CASE 
        WHEN overall_avg_sentiment >= 0.6 THEN '🌟 Excellent'
        WHEN overall_avg_sentiment >= 0.3 THEN '✅ Good'
        WHEN overall_avg_sentiment >= 0 THEN '📊 Fair'
        ELSE '⚠️ Needs Improvement'
    END AS overall_grade
FROM exec_summary
ORDER BY effectiveness_score DESC;


---

## Paso 9: Executive Voice & Tone Consistency

### Qué Vamos a Hacer

Analyze **consistency of voice and tone** across different executives and message types.

### Why This Matters

- **Brand alignment**: Ensure consistent executive messaging
- **Identify outliers**: Find messages that don't match brand voice
- **Training opportunities**: Coach executives on tone consistency

### Consistency Checks

- Sentiment consistency across message types
- Tone alignment with company values
- Individual vs organizational voice

In [ ]:

-- Executive Voice & Tone Consistency
WITH executive_voice AS (
    SELECT 
        r.exec_name,
        COUNT(*) AS total_communications,
        ROUND(AVG(s.sentiment_score), 3) AS exec_avg_sentiment,
        ROUND(STDDEV(s.sentiment_score), 3) AS exec_sentiment_std,
        ROUND(AVG(s.sentiment_magnitude), 3) AS exec_avg_intensity,
        ROUND(MIN(s.sentiment_score), 3) AS min_sentiment,
        ROUND(MAX(s.sentiment_score), 3) AS max_sentiment
    FROM executive_communications r
    JOIN communication_sentiment s ON r.message_id = s.message_id
    GROUP BY r.exec_name
),
org_baseline AS (
    SELECT 
        ROUND(AVG(sentiment_score), 3) AS org_avg_sentiment,
        ROUND(STDDEV(sentiment_score), 3) AS org_sentiment_std
    FROM communication_sentiment
),
voice_comparison AS (
    SELECT 
        ev.*,
        ob.org_avg_sentiment,
        ob.org_sentiment_std,
        ROUND(ev.exec_avg_sentiment - ob.org_avg_sentiment, 3) AS sentiment_vs_org,
        ROUND(ev.exec_sentiment_std / NULLIF(ob.org_sentiment_std, 0), 2) AS volatility_ratio
    FROM executive_voice ev
    CROSS JOIN org_baseline ob
)
SELECT 
    exec_name,
    total_communications,
    exec_avg_sentiment,
    org_avg_sentiment,
    sentiment_vs_org,
    exec_sentiment_std,
    ROUND(max_sentiment - min_sentiment, 3) AS sentiment_range,
    volatility_ratio,
    ROUND(GREATEST(0, 100 - (exec_sentiment_std * 100)), 1) AS consistency_score,
    CASE 
        WHEN ABS(sentiment_vs_org) <= 0.1 THEN '🎯 Perfectly Aligned'
        WHEN ABS(sentiment_vs_org) <= 0.2 THEN '✅ Well Aligned'
        ELSE '⚠️ Misaligned'
    END AS alignment_status,
    CASE 
        WHEN exec_sentiment_std <= 0.2 THEN '💎 Highly Consistent'
        WHEN exec_sentiment_std <= 0.4 THEN '✅ Consistent'
        WHEN exec_sentiment_std <= 0.6 THEN '📊 Variable'
        ELSE '⚠️ Inconsistent Tone'
    END AS tone_consistency,
    CASE 
        WHEN sentiment_vs_org > 0.2 THEN '😊 More Positive than Org'
        WHEN sentiment_vs_org < -0.2 THEN '😐 More Reserved than Org'
        ELSE '➡️ Matches Org Tone'
    END AS voice_positioning
FROM voice_comparison
ORDER BY consistency_score DESC, total_communications DESC;


---

## Paso 10: Strategic Communication Trends

### Qué Vamos a Hacer

Track **strategic themes and priorities** in executive communications over time.

### Why This Matters

- **Strategy alignment**: Ensure executive communications support strategic priorities
- **Trend identification**: Spot emerging themes and shifts
- **Gap analysis**: Find under-communicated strategic initiatives

### Trend Analysis

- Theme evolution over time
- Priority shifts in messaging
- Alignment with organizational goals
- Strategic communication gaps

In [ ]:

-- Strategic Communication Trends
WITH weekly_view AS (
    SELECT 
        DATE_TRUNC('week', r.created_timestamp) AS week_start,
        r.exec_name,
        r.message_type,
        r.topic,
        COUNT(*) AS message_volume,
        ROUND(AVG(s.sentiment_score), 3) AS avg_sentiment,
        ROUND(AVG(s.sentiment_magnitude), 3) AS avg_intensity
    FROM executive_communications r
    JOIN communication_sentiment s ON r.message_id = s.message_id
    GROUP BY week_start, r.exec_name, r.message_type, r.topic
),
trend_analysis AS (
    SELECT 
        w.*,
        LAG(message_volume) OVER (PARTITION BY exec_name, message_type, topic ORDER BY week_start) AS prev_volume,
        LAG(avg_sentiment) OVER (PARTITION BY exec_name, message_type, topic ORDER BY week_start) AS prev_sentiment
    FROM weekly_view w
),
topic_rank AS (
    SELECT 
        week_start,
        topic,
        SUM(message_volume) AS total_volume,
        RANK() OVER (PARTITION BY week_start ORDER BY SUM(message_volume) DESC) AS topic_rank,
        COUNT(DISTINCT exec_name) AS execs_contributing
    FROM weekly_view
    WHERE topic IS NOT NULL
    GROUP BY week_start, topic
)
SELECT 
    ta.week_start,
    ta.exec_name,
    ta.topic,
    ta.message_type,
    ta.message_volume,
    (ta.message_volume - prev_volume) AS volume_change,
    ta.avg_sentiment,
    ROUND(ta.avg_sentiment - prev_sentiment, 3) AS sentiment_change,
    tr.total_volume AS topic_total_volume,
    tr.execs_contributing,
    tr.topic_rank,
    CASE 
        WHEN (ta.message_volume - prev_volume) > 3 THEN '📈 Surging'
        WHEN (ta.message_volume - prev_volume) > 0 THEN '↗️ Growing'
        WHEN (ta.message_volume - prev_volume) < -3 THEN '📉 Declining'
        WHEN (ta.message_volume - prev_volume) < 0 THEN '↘️ Decreasing'
        ELSE '➡️ Stable'
    END AS volume_trend,
    CASE 
        WHEN (ta.avg_sentiment - prev_sentiment) > 0.2 THEN '😊 Improving'
        WHEN (ta.avg_sentiment - prev_sentiment) > 0 THEN '🙂 Slightly Up'
        WHEN (ta.avg_sentiment - prev_sentiment) < -0.2 THEN '😟 Worsening'
        WHEN (ta.avg_sentiment - prev_sentiment) < 0 THEN '😐 Slightly Down'
        ELSE '➡️ Unchanged'
    END AS sentiment_trend,
    CASE 
        WHEN tr.topic_rank <= 3 AND tr.execs_contributing >= 3 THEN '🎯 Top Priority'
        WHEN tr.topic_rank <= 5 THEN '✅ High Priority'
        WHEN tr.topic_rank <= 10 THEN '📊 Moderate Priority'
        ELSE '💭 Emerging Topic'
    END AS strategic_importance
FROM trend_analysis ta
LEFT JOIN topic_rank tr ON ta.week_start = tr.week_start AND ta.topic = tr.topic
WHERE ta.week_start >= DATEADD('week', -8, CURRENT_DATE())
ORDER BY ta.week_start DESC, tr.topic_rank, ta.message_volume DESC
LIMIT 50;


In [ ]:
-- Calculate daily metrics
CREATE OR REPLACE TABLE communication_metrics AS
SELECT 
    'METRIC_' || LPAD(ROW_NUMBER() OVER (ORDER BY r.created_timestamp::DATE)::VARCHAR, 8, '0') AS metric_id,
    r.created_timestamp::DATE AS metric_date,
    'daily_avg_sentiment' AS metric_name,
    ROUND(AVG(s.sentiment_score), 3) AS metric_value,
    CASE 
        WHEN AVG(s.sentiment_score) > LAG(AVG(s.sentiment_score)) OVER (ORDER BY r.created_timestamp::DATE) THEN 'up'
        WHEN AVG(s.sentiment_score) < LAG(AVG(s.sentiment_score)) OVER (ORDER BY r.created_timestamp::DATE) THEN 'down'
        ELSE 'stable'
    END AS trend,
    CURRENT_TIMESTAMP() AS generated_timestamp
FROM executive_communications r
JOIN communication_sentiment s ON r.message_id = s.message_id
GROUP BY r.created_timestamp::DATE;

SELECT * FROM communication_metrics ORDER BY metric_date DESC LIMIT 10;


---

## Paso 11: Dashboard Interactivo

### Dashboard Features

- **Key metrics**: Summary statistics
- **Trend visualization**: Time-series charts
- **Category breakdown**: Distribution analysis
- **Detail view**: Record-level data

In [ ]:

# Executive Communication Optimizer Dashboard
import streamlit as st
import pandas as pd
import altair as alt
from snowflake.snowpark.context import get_active_session

session = get_active_session()
st.title("📊 Executive Communication Optimizer")
st.caption("Powered by Snowflake Cortex AI — Sentiment + Theme intelligence for executives")

required_tables = [
    'EXECUTIVE_COMMUNICATIONS',
    'COMMUNICATION_SENTIMENT',
    'COMMUNICATION_THEMES',
    'COMMUNICATION_METRICS'
]
missing = []
for table in required_tables:
    try:
        session.sql(f"SELECT 1 FROM {table} LIMIT 1").collect()
    except Exception:
        missing.append(table)

if missing:
    st.error("Required tables missing: " + ", ".join(missing))
    st.info("Run the SQL cells above to create and populate the data model before launching the dashboard.")
    st.stop()

summary = session.sql("""
    SELECT 
        (SELECT COUNT(*) FROM EXECUTIVE_COMMUNICATIONS) AS messages,
        (SELECT COUNT(DISTINCT exec_name) FROM EXECUTIVE_COMMUNICATIONS) AS executives,
        (SELECT ROUND(AVG(sentiment_score), 3) FROM COMMUNICATION_SENTIMENT) AS avg_sentiment,
        (SELECT COUNT(*) FROM COMMUNICATION_THEMES) AS insights
""").to_pandas().iloc[0]

col1, col2, col3, col4 = st.columns(4)
col1.metric("Messages", f"{int(summary['MESSAGES']):,}")
col2.metric("Executives", int(summary['EXECUTIVES']))
col3.metric("Avg Sentiment", summary['AVG_SENTIMENT'])
col4.metric("LLM Insights", int(summary['INSIGHTS']))

st.markdown("---")
tab1, tab2, tab3 = st.tabs(["📈 Tone Trends", "🧠 LLM Themes", "🔍 Raw Communications"])

with tab1:
    st.subheader("Daily Tone Metrics")
    metrics_df = session.sql("""
        SELECT TO_CHAR(metric_date, 'YYYY-MM-DD') AS metric_date, metric_value, trend
        FROM COMMUNICATION_METRICS
        ORDER BY metric_date
    """).to_pandas()
    if metrics_df.empty:
        st.info("Run the metric calculation cell to populate COMMUNICATION_METRICS.")
    else:
        metrics_df['METRIC_DATE'] = pd.to_datetime(metrics_df['METRIC_DATE'], errors='coerce')
        chart = alt.Chart(metrics_df).mark_line(point=True).encode(
            x=alt.X('METRIC_DATE:T', title='Date'),
            y=alt.Y('METRIC_VALUE:Q', title='Avg Sentiment'),
            color=alt.value('#2E86DE'),
            tooltip=[alt.Tooltip('METRIC_DATE:T', title='Date'), alt.Tooltip('METRIC_VALUE:Q', title='Sentiment'), 'TREND']
        ).properties(height=350)
        st.altair_chart(chart, use_container_width=True)
        st.dataframe(metrics_df, use_container_width=True)

with tab2:
    st.subheader("AI Themes & Recommendations")
    themes_df = session.sql("""
        SELECT exec_name, topic, key_theme, recommended_action, severity_flag, generated_timestamp
        FROM COMMUNICATION_THEMES
        ORDER BY generated_timestamp DESC
        LIMIT 150
    """).to_pandas()
    if themes_df.empty:
        st.info("Run the LLM theme extraction cell to populate COMMUNICATION_THEMES.")
    else:
        st.dataframe(themes_df, use_container_width=True)
        csv = themes_df.to_csv(index=False)
        st.download_button("Download Themes CSV", csv, "communication_themes.csv")

with tab3:
    st.subheader("Detailed Communications")
    limit = st.slider("Records", 10, 200, 50)
    detail_df = session.sql(f"""
        SELECT ec.created_timestamp, ec.exec_name, ec.message_type, ec.audience, ec.topic,
               s.sentiment_label, s.sentiment_score, ec.content
        FROM EXECUTIVE_COMMUNICATIONS ec
        JOIN COMMUNICATION_SENTIMENT s ON ec.message_id = s.message_id
        ORDER BY ec.created_timestamp DESC
        LIMIT {limit}
    """).to_pandas()
    if detail_df.empty:
        st.info("No communications available. Run the data generation cells first.")
    else:
        st.dataframe(detail_df, use_container_width=True)
        csv = detail_df.to_csv(index=False)
        st.download_button("Download Messages CSV", csv, "executive_messages.csv")

st.markdown("---")
st.success("✅ Dashboard operational | Data current")


---

##  Tutorial Complete!

### What You've Learned

1.  **AI-powered analysis** with Snowflake Cortex
2.  **Production data pipelines** with SQL
3.  **Aggregation and metrics** calculation
4.  **Trend analysis** with window functions
5.  **Interactive dashboards** with Streamlit

### Key Techniques

- **`SENTIMENT() and COMPLETE()`** for AI processing
- **Window functions** for trends
- **CTEs** for complex logic
- **Aggregations** for insights
- **Streamlit** for visualization

### Next Steps for Production

1. **Connect real data sources**
2. **Automate data refresh**
3. **Add alerting logic**
4. **Scale to production volumes**
5. **Integrate with reporting tools**

---

**Congratulations!** You've built a production-ready executive communication optimizer & tone analyzer using Snowflake Cortex AI and SQL.

**Estimated runtime**: (varies by data size)

---

## Limpieza del Entorno (Opcional)

### Qué Hace Esto

This cell will **completely remove** all objects created in this tutorial:
- Drops the `EXEC_COMM_DB` database (and all tables/models within it)
- Drops the `COMPUTE_WH` warehouse

### When to Use

Run this if you want to:
- Clean up your Snowflake environment after completing the tutorial
- Start fresh and re-run the entire notebook
- Remove all demo data

### Instructions

**This cell is commented out by default** to prevent accidental deletion when running "Run All".

To reset your environment:
1. **Remove the comment markers** (`--`) from the SQL commands below
2. **Run this cell manually**

 **Warning**: This action cannot be undone. All data and models will be permanently deleted.

In [ ]:
-- Descomentar las líneas siguientes and ejecutar esta celda para limpiar el entorno

-- DROP DATABASE IF EXISTS EXEC_COMM_DB;
-- DROP WAREHOUSE IF EXISTS COMPUTE_WH;
